#Section 1: Environment Setup & Reproducibility
Import necessary dependencies (transformers, datasets, torch, scikit-learn).

Set fixed random seeds (SEED = 42) across PyTorch, NumPy, and Python's random module to ensure experiment reproducibility.

In [ ]:
import transformers
import datasets
import sklearn
import torch

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

Transformers: 5.16.1
Datasets: 4.8.5
Scikit-learn: 1.6.1
PyTorch: 2.11.0+cu128
GPU: True


In [ ]:
import numpy as np
import pandas as pd
import torch
import time

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#Section 2: Dataset Loading & Label Mapping
Load the Universal Dependencies English EWT dataset (universal-dependencies/universal_dependencies with en_ewt configuration).

Extract the 17 Universal POS (upos) tags, building explicit id2label and label2id mapping dictionaries

In [ ]:
from datasets import load_dataset

ds = load_dataset("universal-dependencies/universal_dependencies", "en_ewt")
ds

README.md:   0%|          | 0.00/90.7k [00:00<?, ?B/s]

parquet/en_ewt/dev.parquet: reconstructing file:   0%|          |  0.00B /  516kB            

parquet/en_ewt/dev.parquet: downloading bytes:           |  0.00B            

parquet/en_ewt/test.parquet: reconstructing file:   0%|          |  0.00B /  523kB            

parquet/en_ewt/test.parquet: downloading bytes:           |  0.00B            

...

parquet/en_ewt/train.parquet: downloading bytes:           |  0.00B            

Generating dev split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})

#Section 3: Tokenization & Subword Label Alignment
1. Rationale: Language models like BERT use subword tokenizers (e.g., WordPiece). A single word may split into multiple subtokens (e.g., "sleeping" $\rightarrow$ "sleep", "##ing").
2. Alignment Strategy:Assign the target POS tag only to the first subtoken of a word.Set label ID -100 to all subsequent subtokens (word_idx == previous_word_idx), special tokens ([CLS], [SEP]), and padding tokens.
3. PyTorch's CrossEntropyLoss automatically ignores -100, preventing subword fragmentation from skewing evaluation metrics and gradients

In [ ]:
print(ds["train"][0])

{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.', 'comments': ['newdoc id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000', '__SENT_ID__', 'newpar id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-p0001', '__TEXT__'], 'tokens': ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'lemmas': ['Al', '-', 'Zaman', ':', 'American', 'force', 'kill', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'upos': ['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT

In [ ]:
print(ds["train"].features)

{'sent_id': Value('string'), 'text': Value('string'), 'comments': List(Value('string')), 'tokens': List(Value('string')), 'lemmas': List(Value('string')), 'upos': List(Value('string')), 'xpos': List(Value('string')), 'feats': List(Value('string')), 'head': List(Value('int16')), 'deprel': List(Value('string')), 'deps': List(Value('string')), 'misc': List(Value('string')), 'mwt': List({'id': Value('string'), 'form': Value('string'), 'feats': Value('string'), 'misc': Value('string')}), 'empty_nodes': List({'id': Value('string'), 'form': Value('string'), 'lemma': Value('string'), 'upos': Value('string'), 'xpos': Value('string'), 'feats': Value('string'), 'head': Value('string'), 'deprel': Value('string'), 'deps': Value('string'), 'misc': Value('string')})}


In [ ]:
label_feature = ds["train"].features["upos"]

print(label_feature)

List(Value('string'))


In [ ]:
label_names = sorted(set(
    label
    for sentence in ds["train"]["upos"]
    for label in sentence
))

print(label_names)
print("Number of labels:", len(label_names))

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
Number of labels: 17


In [ ]:
id2label = {
    i: label
    for i, label in enumerate(label_names)
}

label2id = {
    label: i
    for i, label in enumerate(label_names)
}

print(id2label)

{0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ', 5: 'DET', 6: 'INTJ', 7: 'NOUN', 8: 'NUM', 9: 'PART', 10: 'PRON', 11: 'PROPN', 12: 'PUNCT', 13: 'SCONJ', 14: 'SYM', 15: 'VERB', 16: 'X'}


In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["upos"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label_names.index(labels[word_idx]))

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [ ]:
tokenized_dataset = ds.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12544
    })
})

In [ ]:
example = tokenized_dataset["train"][0]

tokens = tokenizer.convert_ids_to_tokens(
    example["input_ids"]
)

labels = example["labels"]

for token, label in zip(tokens, labels):
    print(f"{token:15} {label:3} {id2label[label] if label != -100 else '-100'}")

[CLS]           -100 -100
al               11 PROPN
-                12 PUNCT
za               11 PROPN
##man           -100 -100
:                12 PUNCT
american          0 ADJ
forces            7 NOUN
killed           15 VERB
sha              11 PROPN
##ikh           -100 -100
abdullah         11 PROPN
al               11 PROPN
-                12 PUNCT
an               11 PROPN
##i             -100 -100
,                12 PUNCT
the               5 DET
preacher          7 NOUN
at                1 ADP
the               5 DET
mosque            7 NOUN
in                1 ADP
the               5 DET
town              7 NOUN
of                1 ADP
q                11 PROPN
##ai            -100 -100
##m             -100 -100
,                12 PUNCT
near              1 ADP
the               5 DET
syrian            0 ADJ
border            7 NOUN
.                12 PUNCT
[SEP]           -100 -100


#Section 4: Experiment 1 – Feature-based Approach
Freeze BERT's encoder parameters (torch.no_grad()).

Extract the last_hidden_state representations for valid token positions (excluding -100).

Train a scikit-learn LogisticRegression classifier on top of the static embeddings.

Record training duration, execution time, Macro F1, and Accuracy.

In [ ]:
bert = AutoModel.from_pretrained(MODEL_NAME)

bert.to("cuda" if torch.cuda.is_available() else "cpu")

bert.eval()

for param in bert.parameters():
    param.requires_grad = False

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
def extract_features(dataset_split):

    X = []
    y = []

    for example in dataset_split:

        input_ids = torch.tensor(
            example["input_ids"]
        ).unsqueeze(0).to(device)

        attention_mask = torch.tensor(
            example["attention_mask"]
        ).unsqueeze(0).to(device)

        labels = example["labels"]

        with torch.no_grad():

            outputs = bert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        hidden_states = outputs.last_hidden_state[0]

        for embedding, label in zip(hidden_states, labels):

            if label != -100:

                X.append(
                    embedding.cpu().numpy()
                )

                y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = extract_features(
    tokenized_dataset["train"]
)

print(X_train.shape)
print(y_train.shape)

(204578, 768)
(204578,)


In [ ]:
X_val, y_val = extract_features(
    tokenized_dataset["dev"]
)

print(X_val.shape)
print(y_val.shape)

(25148, 768)
(25148,)


In [ ]:
start_time = time.time()

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    n_jobs=-1
)

lr_model.fit(
    X_train,
    y_train
)

training_time = time.time() - start_time

print("Training time:", training_time, "seconds")

Training time: 271.7722363471985 seconds


In [ ]:
y_pred = lr_model.predict(X_val)

In [ ]:
accuracy = accuracy_score(
    y_val,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.9416653411802132


In [ ]:
f1 = f1_score(
    y_val,
    y_pred,
    average="macro"
)

print("Macro F1:", f1)

Macro F1: 0.8935887332239401


In [ ]:
print(
    classification_report(
        y_val,
        y_pred,
        target_names=label_names
    )
)

              precision    recall  f1-score   support

         ADJ       0.89      0.88      0.88      1865
         ADP       0.96      0.95      0.95      2038
         ADV       0.88      0.87      0.88      1232
         AUX       0.97      0.98      0.98      1567
       CCONJ       0.99      0.99      0.99       780
         DET       0.97      0.98      0.98      1900
        INTJ       0.91      0.81      0.86       115
        NOUN       0.91      0.93      0.92      4210
         NUM       0.96      0.97      0.96       383
        PART       0.97      0.98      0.97       647
        PRON       0.98      0.99      0.98      2225
       PROPN       0.90      0.87      0.88      1867
       PUNCT       0.99      0.99      0.99      3075
       SCONJ       0.92      0.91      0.92       397
         SYM       0.68      0.79      0.73        81
        VERB       0.95      0.94      0.94      2707
           X       0.43      0.34      0.38        59

    accuracy              

#Section 5: Experiment 2 – Full Fine-Tuning with Differentiated Learning Rates

Instantiate AutoModelForTokenClassification using bert-base-uncased.

Parameter Grouping & Optimizer Strategy:
- Encoder Parameters (model.bert): Assigned a lower learning rate ($2\times 10^{-5}$) to preserve pre-trained language knowledge and prevent catastrophic forgetting.
- Classifier Head (model.classifier): Assigned a higher learning rate ($1\times 10^{-3}$) because its weights are initialized randomly and require faster convergence.

Instantiate AdamW with grouped parameters and pass it explicitly to Trainer(..., optimizers=(optimizer, None))

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
def compute_metrics(eval_prediction):

    predictions, labels = eval_prediction

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        for p, l in zip(prediction, label):

            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    accuracy = accuracy_score(
        true_labels,
        true_predictions
    )

    f1 = f1_score(
        true_labels,
        true_predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./pos_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True
)

In [ ]:
from torch.optim import AdamW


encoder_params = [p for n, p in model.bert.named_parameters()]
head_params = [p for n, p in model.classifier.named_parameters()]


optimizer_grouped_parameters = [
    {"params": encoder_params, "lr": 2e-5, "weight_decay": 0.01},
    {"params": head_params, "lr": 1e-3, "weight_decay": 0.01}
]


optimizer = AdamW(optimizer_grouped_parameters)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
    )

In [ ]:
start_time = time.time()

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.086076,0.104867,0.970415,0.904355
2,0.044164,0.102791,0.972801,0.924395
3,0.026075,0.110305,0.974352,0.931155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=2352, training_loss=0.08638704329931816, metrics={'train_runtime': 507.6439, 'train_samples_per_second': 74.131, 'train_steps_per_second': 4.633, 'total_flos': 1000714463827296.0, 'train_loss': 0.08638704329931816, 'epoch': 3.0})

In [ ]:
training_time_ft = time.time() - start_time

print(
    "Fine-tuning time:",
    training_time_ft,
    "seconds"
)

Fine-tuning time: 508.23853302001953 seconds


In [ ]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.026075,0.110305,3,0.974352,0.931155


{'eval_loss': 0.11030515283346176,
 'eval_accuracy': 0.9743518371242246,
 'eval_macro_f1': 0.931154897726531}

#Section 6: Model Evaluation & Method Comparison
Compute and log Macro F1, Accuracy, and overall execution time.

Build a comparison table (pd.DataFrame) contrasting the Feature-based method against Full Fine-tuning.

Perform qualitative inference checks on sample sentences, filtering out special tokens ([CLS], [SEP]).


Save the fine-tuned model and tokenizer locally (./final_pos_model).

In [ ]:
comparison = pd.DataFrame({
    "Method": [
        "Feature-based + Logistic Regression",
        "Full Fine-tuning"
    ],
    "Accuracy": [
        accuracy,
        results["eval_accuracy"]
    ],
    "Macro F1": [
        f1,
        results["eval_macro_f1"]
    ],
    "Training Time (seconds)": [
        training_time,
        training_time_ft
    ]
})

comparison

,Method,Accuracy,Macro F1,Training Time (seconds)
0,Feature-based + Logistic Regression,0.941665,0.893589,271.772236
1,Full Fine-tuning,0.974352,0.931155,508.238533


In [ ]:
trainer.save_model("./final_pos_model")
tokenizer.save_pretrained("./final_pos_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_pos_model/tokenizer_config.json', './final_pos_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "./final_pos_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForTokenClassification.from_pretrained(
    model_path
)

print("Modelo cargado correctamente")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado correctamente


In [ ]:
import torch

text = "The cat is sleeping."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(
    outputs.logits,
    dim=-1
)[0]

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

for token, prediction in zip(tokens, predictions):
    label = model.config.id2label[prediction.item()]
    print(f"{token:15} {label}")

[CLS]           PRON
the             DET
cat             NOUN
is              AUX
sleeping        VERB
.               PUNCT
[SEP]           PUNCT


In [ ]:
def predict_pos(text):

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(
        outputs.logits,
        dim=-1
    )[0]

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    results = []

    for token, prediction in zip(tokens, predictions):

        if token in ["[CLS]", "[SEP]"]:
            continue

        label = model.config.id2label[
            prediction.item()
        ]

        results.append((token, label))

    return results

In [ ]:
sentences = [
    "The cat is sleeping.",
    "John bought a new computer.",
    "The students are working on a project.",
    "I really like this movie.",
    "Tomorrow we will visit the museum."
]

for sentence in sentences:

    print("\n" + "=" * 50)
    print(sentence)
    print("=" * 50)

    results = predict_pos(sentence)

    for token, label in results:
        print(f"{token:15} {label}")


The cat is sleeping.
the             DET
cat             NOUN
is              AUX
sleeping        VERB
.               PUNCT

John bought a new computer.
john            PROPN
bought          VERB
a               DET
new             ADJ
computer        NOUN
.               PUNCT

The students are working on a project.
the             DET
students        NOUN
are             AUX
working         VERB
on              ADP
a               DET
project         NOUN
.               PUNCT

I really like this movie.
i               PRON
really          ADV
like            VERB
this            DET
movie           NOUN
.               PUNCT

Tomorrow we will visit the museum.
tomorrow        NOUN
we              PRON
will            AUX
visit           VERB
the             DET
museum          NOUN
.               PUNCT


#Section 7: Model Export & Hugging Face Hub Publication

Generate an automated README.md (Model Card) containing task metadata, performance metrics, limitations, and references.

Programmatically upload artifacts using HfApi to Hugging Face Hub

In [49]:
from huggingface_hub import login, whoami, HfApi
from getpass import getpass
from pathlib import Path


login(token=getpass('HF write token (input hidden): '), add_to_git_credential=False)


repo_id = f"{whoami()['name']}/tds-bert-pos-ewt"
private = True


eval_loss = 0.110305
eval_acc = 0.974352
eval_f1 = 0.931155

BASE = "bert-base-uncased"
DATASET = "universal-dependencies/universal_dependencies"
winner = "Full Fine-tuning with differentiated learning rates"
EPOCHS = 3
TRAIN_BS = 16
MAX_LEN = 128
SEED = 42


card = f"""---
language: en
pipeline_tag: token-classification
tags:
- bert
- pos-tagging
- universal-dependencies
- ewt
metrics:
- accuracy
- f1
model-index:
- name: {repo_id.split('/')[-1]}
  results:
  - task:
      type: token-classification
      name: Part-of-Speech Tagging
    dataset:
      name: UD English EWT
      type: universal-dependencies/universal_dependencies
      config: en_ewt
    metrics:
    - name: Accuracy
      type: accuracy
      value: {eval_acc:.6f}
    - name: Macro F1
      type: f1
      value: {eval_f1:.6f}
---
# BERT POS Tagging on UD English EWT (Winning Model)

## Intended Use
English Part-of-Speech (POS) tagging using Universal Dependencies EWT (17 UPOS tags) developed for an academic course assignment. Not intended for production-critical NLP applications without domain-specific validation.

## Model and Training Details
- **Base Model:** [{BASE}](https://huggingface.co/{BASE})
- **Dataset:** [{DATASET}](https://huggingface.co/datasets/{DATASET}) (`en_ewt`).
- **Adaptation Strategy:** {winner}; random seed {SEED}; {EPOCHS} epochs; batch size {TRAIN_BS}.
- **Optimizer Configuration:** Discriminative learning rates (Encoder LR: 2e-5, Classifier Head LR: 1e-3).
- **Label Alignment:** Supervised the first subtoken only; assigned -100 to subword continuations and special tokens.

## Evaluation Results
Evaluation performed on the validation set (`dev` split):

- **Validation Loss:** {eval_loss:.6f}
- **Validation Accuracy:** {eval_acc:.6f} ({eval_acc * 100:.2f}%)
- **Validation Macro F1:** {eval_f1:.6f}

### Comparison with Baseline
| Model Variant | Accuracy | Macro F1 |
| :--- | :---: | :---: |
| Feature-based Baseline (Logistic Regression) | 0.941665 | 0.893589 |
| **Full Fine-tuning (This Model)** | **{eval_acc:.6f}** | **{eval_f1:.6f}** |

## Limitations
Trained on English Web Treebank (EWT). Performance may degrade on informal web text, domain-specific terminology, or noisy inputs. Input sequences exceeding maximum sequence length are truncated.

## References
- Universal Dependencies EWT Framework.
- Devlin et al. (2019), [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805).
- [Hugging Face Token Classification Guide](https://huggingface.co/docs/transformers/tasks/token_classification).
"""

model_dir = Path("./final_pos_model")
(model_dir / "README.md").write_text(card, encoding="utf-8")

api = HfApi()
api.create_repo(repo_id=repo_id, private=private, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=str(model_dir), repo_type='model')

print('Published successfully to:', f'https://huggingface.co/{repo_id}')

HF write token (input hidden): ··········
Published successfully to: https://huggingface.co/ramiroc3/tds-bert-pos-ewt
